In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from datetime import datetime
from pyspark.sql.window import Window

# Widget with default value
dbutils.widgets.text("table_name", "fact_sales")
table_name = dbutils.widgets.get("table_name")

# Configuration & Paths
silver_base = "abfss://processed@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/silver/"
gold_base = "abfss://curated@[YOUR_ACCOUNT_STORAGE_NAME].dfs.core.windows.net/gold/"
target_table = table_name
gold_path = f"{gold_base}{target_table}"

# Silver Sources
df_items = spark.read.format("delta").load(f"{silver_base}olist_order_items_dataset")
df_orders = spark.read.format("delta").load(f"{silver_base}olist_orders_dataset")

# Gold Dimension Sources
df_dim_orders = spark.read.format("delta").load(f"{gold_base}dim_orders")
df_dim_customers = spark.read.format("delta").load(f"{gold_base}dim_customers")
df_dim_sellers = spark.read.format("delta").load(f"{gold_base}dim_sellers")
df_dim_products = spark.read.format("delta").load(f"{gold_base}dim_products")

# Function for total order price to allocate freight proportionally per item
window_spec = Window.partitionBy("oi.order_id")

df_sales_base = (df_items.alias("oi")
                 .join(df_orders.alias("o"), F.col("oi.order_id") == F.col("o.order_id"), "left")
                 .withColumn("total_order_price", F.sum("price").over(window_spec))
                 .select(
                     "oi.order_id", "oi.product_id", "oi.seller_id", "o.customer_id", "oi.order_item_id",
                     "o.order_status", "o.order_purchase_timestamp", "o.order_approved_at",
                     "o.order_delivered_carrier_date", "o.order_delivered_customer_date",
                     "o.order_estimated_delivery_date",
                     F.col("oi.price").alias("product_price"),
                     "oi.freight_value",
                     "total_order_price",
                 ))

# Joins
df_enriched = (df_sales_base.alias("sb")
               .join(df_dim_orders.alias("do"), F.col("sb.order_id") == F.col("do.order_id"), "left")
               .join(df_dim_customers.alias("dc"), 
                     (F.col("sb.customer_id") == F.col("dc.customer_id")) &
                     (F.col("sb.order_purchase_timestamp") >= F.col("dc.start_date")) &
                     ((F.col("sb.order_purchase_timestamp") < F.col("dc.end_date")) | (F.col("dc.end_date").isNull())), 
                     "left")
               .join(df_dim_sellers.alias("ds"), 
                     (F.col("sb.seller_id") == F.col("ds.seller_id")) &
                     (F.col("sb.order_purchase_timestamp") >= F.col("ds.start_date")) &
                     ((F.col("sb.order_purchase_timestamp") < F.col("ds.end_date")) | (F.col("ds.end_date").isNull())), 
                     "left")
               .join(df_dim_products.alias("dp"), 
                     (F.col("sb.product_id") == F.col("dp.product_id")) &
                     (F.col("sb.order_purchase_timestamp") >= F.col("dp.start_date")) &
                     ((F.col("sb.order_purchase_timestamp") < F.col("dp.end_date")) | (F.col("ds.end_date").isNull())), 
                     "left"))

df_fact_sales = (df_enriched.select(
                    # Surrogate Key
                    F.md5(F.concat(F.col("sb.order_id"),
                                 F.col("sb.product_id"),
                                 F.col("sb.order_item_id")))
                    .cast("string").alias("sales_key"),

                    # Dimension Keys
                    F.coalesce(F.col("do.order_key"), F.lit("-1")).cast("string").alias("order_key"),
                    F.coalesce(F.col("dc.customer_key"), F.lit("-1")).cast("string").alias("customer_key"),
                    F.coalesce(F.col("ds.seller_key"), F.lit("-1")).cast("string").alias("seller_key"),
                    F.coalesce(F.col("dp.product_key"), F.lit("-1")).cast("string").alias("product_key"),

                    "sb.order_item_id",
                    "sb.product_price",

                    # Freight Allocation Logic
                    F.when(F.col("sb.total_order_price") == 0, 0)
                    .otherwise((F.col("sb.product_price") / F.col("sb.total_order_price")) * F.col("sb.freight_value")).cast("decimal(10,2)").alias("allocated_freight"),
                    
                    # Logistics & Dates
                    "sb.order_status",
                    "sb.order_purchase_timestamp",
                    "sb.order_approved_at",
                    "sb.order_delivered_carrier_date",
                    "sb.order_delivered_customer_date",

                    # Performance Metrics
                    F.datediff("sb.order_delivered_customer_date", "sb.order_purchase_timestamp").alias("total_delivery_days"),
                    F.datediff("sb.order_delivered_carrier_date", "sb.order_purchase_timestamp").alias("seller_processing_days"),
                    F.datediff("sb.order_delivered_customer_date", "sb.order_delivered_carrier_date").alias("carrier_transit_days"),
                    
                    # Flags
                    F.when(F.col("sb.order_delivered_customer_date") > F.col("sb.order_estimated_delivery_date"), 
                        F.datediff("sb.order_delivered_customer_date", "sb.order_estimated_delivery_date"))
                        .otherwise(0).alias("delivery_delay_days"),
                    F.when(F.col("sb.order_delivered_carrier_date").isNotNull(), 1).otherwise(0).alias("is_shipped_flag"),
                    F.when(F.col("sb.order_status") == "delivered", 1).otherwise(0).alias("is_delivered_flag"),
                    F.when(F.col("sb.order_delivered_customer_date") > F.col("sb.order_estimated_delivery_date"), 1).otherwise(0).alias("is_late_delivery_flag"),
                    F.date_trunc("second", F.current_timestamp()).alias("gold_load_at"))
                 .withColumn("total_product_value", F.col("product_price") + F.col("allocated_freight"))
                 )

# Incremental Merge (Upsert)
print(f"--> Starting Gold Load: {target_table}")

if DeltaTable.isDeltaTable(spark, gold_path):
    dt_gold = DeltaTable.forPath(spark, gold_path)

    # Merge Condition
    data_cols = ["order_status", "total_delivery_days", "is_late_delivery_flag"]
    change_condition = " OR ".join([f"NOT (target.{c} <=> source.{c})" for c in data_cols])

    (dt_gold.alias("target")
     .merge(df_fact_sales.alias("source"), "target.sales_key = source.sales_key")
     .whenMatchedUpdateAll(condition = change_condition)
     .whenNotMatchedInsertAll()
     .execute())
    print(f"--> [Merge] {target_table} updated.")

else:
    #First time load
    df_fact_sales.write.format("delta").mode("overwrite").save(gold_path)
    print(f"--> [INIT] {target_table} created.")

    # ZORDERING on order purchase timestamp
    spark.sql(f"OPTIMIZE delta.`{gold_path}` ZORDER BY (order_purchase_timestamp)")

# Audit & Exit
dt_final = DeltaTable.forPath(spark, gold_path)

last_operation = dt_final.history(1).select("operationMetrics").collect()[0][0]
rows_inserted = int(last_operation.get("numTargetRowsInserted", 0))
rows_updated = int(last_operation.get("numTargetRowsUpdated", 0))
total_rows = last_operation.get("numOutputRows", "Check History")

print("-" * 50)
print(f"--> Table: {table_name} | Status: Success")
print(f"--> Rows Processed in last run: {rows_inserted + rows_updated}")
print("-" * 50)

dbutils.notebook.exit(f"Success | Inserted: {rows_inserted} | Updated: {rows_updated}")